In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.signal import butter, filtfilt
from google.colab import drive
import zipfile

drive.mount('/content/drive')

# unzip only if not already done
zip_path = '/content/drive/MyDrive/internship.zip'
if not os.path.exists('/content/internship/internship/Data'):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/internship')
    print("extracted!")
else:
    print("already extracted")

# paths
BASE     = '/content/internship/internship'
DATA_DIR = BASE + '/Data'
VIZ_DIR  = BASE + '/Visualizations'
DS_DIR   = BASE + '/Dataset'

os.makedirs(VIZ_DIR, exist_ok=True)
os.makedirs(DS_DIR,  exist_ok=True)
os.makedirs(BASE + '/scripts', exist_ok=True)


# helper to find the right file inside a participant folder
# needed because filenames are slightly different for each participant
def find_file(folder, keywords, exclude=None):
    for fname in os.listdir(folder):
        f = fname.lower()
        if all(k.lower() in f for k in keywords):
            if exclude and any(e.lower() in f for e in exclude):
                continue
            return os.path.join(folder, fname)
    return None


# reads one of the signal txt files (flow/thorac/spo2)
# skips all the header lines and parses timestamp ; value
def parse_signal(filepath):
    timestamps, values = [], []
    reading = False
    with open(filepath, 'r', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if line.lower() == 'data:':
                reading = True
                continue
            if reading and ';' in line:
                parts = line.split(';')
                try:
                    ts  = pd.to_datetime(parts[0].strip(), format='%d.%m.%Y %H:%M:%S,%f')
                    val = float(parts[1].strip())
                    timestamps.append(ts)
                    values.append(val)
                except:
                    pass  # skip malformed lines
    return pd.Series(values, index=pd.DatetimeIndex(timestamps), name='value')


# parses the flow events file
# each line looks like: 30.05.2024 23:48:45,119-23:49:01,408; 16; Hypopnea; N1
def parse_events(filepath):
    events = []
    with open(filepath, 'r', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if ';' not in line or '-' not in line.split(';')[0]:
                continue
            try:
                parts      = line.split(';')
                time_range = parts[0].strip()
                event_type = parts[2].strip()

                date_str        = time_range[:10]   # DD.MM.YYYY
                start_str, end_str = time_range[11:].split('-')

                start = pd.to_datetime(date_str + ' ' + start_str, format='%d.%m.%Y %H:%M:%S,%f')
                end   = pd.to_datetime(date_str + ' ' + end_str,   format='%d.%m.%Y %H:%M:%S,%f')

                # handle midnight crossover (e.g event starts 23:59 ends 00:01)
                if end < start:
                    end += pd.Timedelta(days=1)

                events.append({'start': start, 'end': end, 'event': event_type})
            except:
                pass
    return pd.DataFrame(events)


# bandpass filter — keeps only breathing frequency range (0.17–0.4 Hz)
# anything outside that range is noise
def bandpass_filter(sig, fs, lo=0.17, hi=0.4, order=4):
    nyq  = fs / 2.0
    b, a = butter(order, [lo/nyq, hi/nyq], btype='band')
    return filtfilt(b, a, sig)


# figures out the label for a 30s window
# if more than 50% of the window overlaps with a known event → use that label
# otherwise → Normal
def get_label(win_start, win_end, events_df):
    duration = (win_end - win_start).total_seconds()
    for _, ev in events_df.iterrows():
        overlap = (min(win_end, ev['end']) - max(win_start, ev['start'])).total_seconds()
        if overlap > 0.5 * duration:
            return ev['event']
    return 'Normal'


def plot_participant(folder, out_dir):

    flow_f   = find_file(folder, ['flow'],   exclude=['events'])
    thorac_f = find_file(folder, ['thorac'])
    spo2_f   = find_file(folder, ['spo2'])
    events_f = find_file(folder, ['flow', 'events'])

    if not all([flow_f, thorac_f, spo2_f, events_f]):
        print(f"missing files in {folder}, skipping")
        return

    flow   = parse_signal(flow_f)
    thorac = parse_signal(thorac_f)
    spo2   = parse_signal(spo2_f)
    events = parse_events(events_f)

    pid = os.path.basename(folder)

    # downsample for plotting otherwise its too dense
    # flow/thorac at 32Hz → take every 32nd = 1 sample/sec
    # spo2 at 4Hz → take every 4th = 1 sample/4sec
    flow_ds   = flow.iloc[::32]
    thorac_ds = thorac.iloc[::32]
    spo2_ds   = spo2.iloc[::4]

    colors = {
        'Hypopnea':          '#FF8C00',
        'Obstructive Apnea': '#CC0000',
        'Central Apnea':     '#9900CC',
        'Mixed Apnea':       '#0066CC',
    }

    fig, axes = plt.subplots(3, 1, figsize=(20, 12), sharex=True)
    fig.suptitle(f'Sleep Study - Participant {pid}', fontsize=16, fontweight='bold', y=0.98)

    for sig, label, color, ax in [
        (flow_ds,   'Nasal Airflow',     'steelblue', axes[0]),
        (thorac_ds, 'Thoracic Movement', 'seagreen',  axes[1]),
        (spo2_ds,   'SpO₂ (%)',          'crimson',   axes[2]),
    ]:
        t0    = sig.index[0]
        hrs   = (sig.index - t0).total_seconds() / 3600.0
        ax.plot(hrs, sig.values, color=color, linewidth=0.4, alpha=0.85)
        ax.set_ylabel(label, fontsize=11)
        ax.grid(True, alpha=0.3)

        # shade the event regions
        for _, ev in events.iterrows():
            s = (ev['start'] - t0).total_seconds() / 3600.0
            e = (ev['end']   - t0).total_seconds() / 3600.0
            ax.axvspan(s, e, alpha=0.35, color=colors.get(ev['event'], '#888'), linewidth=0)

    axes[2].set_xlabel('Time (hours from start)', fontsize=11)

    patches = [mpatches.Patch(color=c, label=ev, alpha=0.6) for ev, c in colors.items()]
    fig.legend(handles=patches, loc='lower center', ncol=4, fontsize=10,
               title='Breathing Events', title_fontsize=10, bbox_to_anchor=(0.5, 0.01))
    plt.tight_layout(rect=[0, 0.05, 1, 0.97])

    os.makedirs(out_dir, exist_ok=True)
    out = os.path.join(out_dir, f'{pid}_visualization.pdf')
    fig.savefig(out, format='pdf', dpi=150, bbox_inches='tight')
    print(f"saved → {out}")
    plt.show()
    plt.close()


print("\nVisualizations ")
for pid in sorted(os.listdir(DATA_DIR)):
    folder = os.path.join(DATA_DIR, pid)
    if os.path.isdir(folder):
        print(f"plotting {pid}...")
        plot_participant(folder, VIZ_DIR)


print("\n Building dataset ")
all_rows = []

for pid in sorted(os.listdir(DATA_DIR)):
    folder = os.path.join(DATA_DIR, pid)
    if not os.path.isdir(folder):
        continue
    print(f"processing {pid}...")

    flow_f   = find_file(folder, ['flow'],   exclude=['events'])
    thorac_f = find_file(folder, ['thorac'])
    spo2_f   = find_file(folder, ['spo2'])
    events_f = find_file(folder, ['flow', 'events'])

    if not all([flow_f, thorac_f, spo2_f, events_f]):
        print(f"  missing files, skipping")
        continue

    flow   = parse_signal(flow_f)
    thorac = parse_signal(thorac_f)
    spo2   = parse_signal(spo2_f)
    events = parse_events(events_f)

    # filter to breathing frequency range
    flow_f2   = bandpass_filter(flow.values,   fs=32)
    thorac_f2 = bandpass_filter(thorac.values, fs=32)
    spo2_f2   = bandpass_filter(spo2.values,   fs=4)

    # 30s windows, 50% overlap
    # at 32Hz: 30s = 960 samples, step = 480
    win  = 960
    step = 480
    n    = 0

    for i in range(0, len(flow_f2) - win + 1, step):
        ws = flow.index[i]
        we = flow.index[i + win - 1]

        # spo2 is at 4Hz so index is 8x slower than 32Hz
        si = i // 8
        se = si + 120
        if se > len(spo2_f2):
            break

        label = get_label(ws, we, events)

        row = {'participant': pid, 'win_start': str(ws), 'label': label}
        for j, v in enumerate(flow_f2[i:i+win]):    row[f'flow_{j}']   = v
        for j, v in enumerate(thorac_f2[i:i+win]):  row[f'thorac_{j}'] = v
        for j, v in enumerate(spo2_f2[si:se]):      row[f'spo2_{j}']   = v

        all_rows.append(row)
        n += 1

    print(f"  {pid}: {n} windows")

df = pd.DataFrame(all_rows)
df.to_csv(DS_DIR + '/breathing_dataset.csv', index=False)

print(f"\ntotal windows : {len(df)}")
print(f"label counts  :\n{df['label'].value_counts()}")
print(f"saved to      : {DS_DIR}/breathing_dataset.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
already extracted

Visualizations 
plotting AP01...


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
import numpy as np
import pandas as pd

# ── Load dataset ───────────────────────────────────────────────
df = pd.read_csv('/content/internship/internship/Dataset/breathing_dataset.csv')

# group rare classes into 'Other' to avoid 1-2 sample issues
df['label'] = df['label'].replace({'Body event': 'Other', 'Mixed Apnea': 'Other'})

print("Label counts after grouping:")
print(df['label'].value_counts())

# ── Encode labels ──────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['label'])
print("\nClasses:", list(le.classes_))

# ── Feature columns ────────────────────────────────────────────
flow_cols   = [c for c in df.columns if c.startswith('flow_')]
thorac_cols = [c for c in df.columns if c.startswith('thorac_')]
spo2_cols   = [c for c in df.columns if c.startswith('spo2_')]

print(f"\nflow: {len(flow_cols)}, thorac: {len(thorac_cols)}, spo2: {len(spo2_cols)}")
print(f"Total features per window: {len(flow_cols)+len(thorac_cols)+len(spo2_cols)}")

Label counts after grouping:
label
Normal               8038
Hypopnea              593
Obstructive Apnea     164
Other                   5
Name: count, dtype: int64

Classes: ['Hypopnea', 'Normal', 'Obstructive Apnea', 'Other']

flow: 960, thorac: 960, spo2: 120
Total features per window: 2040


In [ ]:
# ── Build input tensors ────────────────────────────────────────

flow_data   = df[flow_cols].values.astype(np.float32)    # (8800, 960)
thorac_data = df[thorac_cols].values.astype(np.float32)  # (8800, 960)
spo2_data   = df[spo2_cols].values.astype(np.float32)    # (8800, 120)

# upsample spo2 from 120 → 960 by repeating each value 8 times
# this matches the 32Hz length so all 3 channels are same size
spo2_up = np.repeat(spo2_data, 8, axis=1)                # (8800, 960)

# stack into (8800, 3, 960) — 3 channels, 960 timepoints
X = np.stack([flow_data, thorac_data, spo2_up], axis=1)  # (8800, 3, 960)
y = df['label_enc'].values.astype(np.int64)

participants = df['participant'].values

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Classes: {le.classes_}")

# ── Define 1D CNN ──────────────────────────────────────────────
class CNN1D(nn.Module):
    def __init__(self, num_classes):
        super(CNN1D, self).__init__()

        self.conv_block = nn.Sequential(
            # first conv layer: 3 input channels → 32 filters, kernel size 7
            nn.Conv1d(in_channels=3, out_channels=32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4),   # 960 → 240

            # second conv layer: 32 → 64 filters
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4),   # 240 → 60

            # third conv layer: 64 → 128 filters
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4),   # 60 → 15
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 15, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.conv_block(x)
        x = self.classifier(x)
        return x

# quick shape test
model_test = CNN1D(num_classes=4)
dummy = torch.zeros(8, 3, 960)
out   = model_test(dummy)
print(f"\nCNN output shape (batch=8): {out.shape}")  # should be (8, 4)
print("CNN architecture OK ✅")

X shape: (8800, 3, 960)
y shape: (8800,)
Classes: ['Hypopnea' 'Normal' 'Obstructive Apnea' 'Other']

CNN output shape (batch=8): torch.Size([8, 4])
CNN architecture OK ✅


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

all_pids    = ['AP01', 'AP02', 'AP03', 'AP04', 'AP05']
all_results = []  # stores per-fold metrics

for test_pid in all_pids:
    print(f"\n{'='*50}")
    print(f"Fold: train on all except {test_pid}, test on {test_pid}")

    # ── Split ──────────────────────────────────────────────────
    train_mask = participants != test_pid
    test_mask  = participants == test_pid

    X_train = torch.tensor(X[train_mask])
    y_train = torch.tensor(y[train_mask])
    X_test  = torch.tensor(X[test_mask])
    y_test  = torch.tensor(y[test_mask])

    print(f"  train: {X_train.shape[0]} windows | test: {X_test.shape[0]} windows")

    # ── Class weights to handle imbalance ─────────────────────
    classes     = np.unique(y[train_mask])
    cw          = compute_class_weight('balanced', classes=classes, y=y[train_mask])
    # fill missing classes with 1.0
    weights     = np.ones(4)
    weights[classes] = cw
    class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

    # ── Dataloader ─────────────────────────────────────────────
    train_ds     = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

    # ── Model, loss, optimizer ─────────────────────────────────
    model     = CNN1D(num_classes=4).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    # ── Training loop ──────────────────────────────────────────
    epochs = 15
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        if (epoch + 1) % 5 == 0:
            print(f"  epoch {epoch+1}/{epochs}  loss: {total_loss/len(train_loader):.4f}")

    # ── Evaluation ─────────────────────────────────────────────
    model.eval()
    with torch.no_grad():
        preds = model(X_test.to(device)).argmax(dim=1).cpu().numpy()

    acc  = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, average='macro', zero_division=0)
    rec  = recall_score(y_test, preds, average='macro', zero_division=0)
    cm   = confusion_matrix(y_test, preds, labels=[0,1,2,3])

    print(f"\n  Results for {test_pid}:")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall   : {rec:.4f}")
    print(f"  Confusion Matrix:\n{cm}")

    all_results.append({
        'fold': test_pid, 'accuracy': acc,
        'precision': prec, 'recall': rec, 'cm': cm
    })

# ── Summary across all folds ───────────────────────────────────
print(f"\n{'='*50}")
print("LOPO Cross-Validation Summary")
print(f"{'='*50}")
accs  = [r['accuracy']  for r in all_results]
precs = [r['precision'] for r in all_results]
recs  = [r['recall']    for r in all_results]

for r in all_results:
    print(f"{r['fold']}  acc={r['accuracy']:.4f}  prec={r['precision']:.4f}  rec={r['recall']:.4f}")

print(f"\nMean Accuracy : {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"Mean Precision: {np.mean(precs):.4f} ± {np.std(precs):.4f}")
print(f"Mean Recall   : {np.mean(recs):.4f} ± {np.std(recs):.4f}")
print(f"\nClass labels: {list(le.classes_)}")

Using: cuda

Fold: train on all except AP01, test on AP01
  train: 6978 windows | test: 1822 windows
  epoch 5/15  loss: 1.0266
  epoch 10/15  loss: 0.8287
  epoch 15/15  loss: 0.7357

  Results for AP01:
  Accuracy : 0.6762
  Precision: 0.3696
  Recall   : 0.6229
  Confusion Matrix:
[[  29   32   18    0]
 [ 350 1190  187    0]
 [   2    1   13    0]
 [   0    0    0    0]]

Fold: train on all except AP02, test on AP02
  train: 7031 windows | test: 1769 windows
  epoch 5/15  loss: 0.9635
  epoch 10/15  loss: 0.8080
  epoch 15/15  loss: 0.6743

  Results for AP02:
  Accuracy : 0.7954
  Precision: 0.3899
  Recall   : 0.5241
  Confusion Matrix:
[[  61   76   13    0]
 [ 249 1345   22    0]
 [   1    1    1    0]
 [   0    0    0    0]]

Fold: train on all except AP03, test on AP03
  train: 7104 windows | test: 1696 windows
  epoch 5/15  loss: 0.8507
  epoch 10/15  loss: 0.7362
  epoch 15/15  loss: 0.6507

  Results for AP03:
  Accuracy : 0.3626
  Precision: 0.2513
  Recall   : 0.1690
  C

In [ ]:
# ── vis.py ─────────────────────────────────────────────────────
vis_py = '''
import os, argparse
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def find_file(folder, keywords, exclude=None):
    for fname in os.listdir(folder):
        f = fname.lower()
        if all(k.lower() in f for k in keywords):
            if exclude and any(e.lower() in f for e in exclude):
                continue
            return os.path.join(folder, fname)
    return None

def parse_signal(filepath):
    timestamps, values = [], []
    reading = False
    with open(filepath, 'r', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if line.lower() == 'data:':
                reading = True
                continue
            if reading and ';' in line:
                parts = line.split(';')
                try:
                    ts  = pd.to_datetime(parts[0].strip(), format='%d.%m.%Y %H:%M:%S,%f')
                    val = float(parts[1].strip())
                    timestamps.append(ts)
                    values.append(val)
                except:
                    pass
    return pd.Series(values, index=pd.DatetimeIndex(timestamps), name='value')

def parse_events(filepath):
    events = []
    with open(filepath, 'r', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if ';' not in line or '-' not in line.split(';')[0]:
                continue
            try:
                parts      = line.split(';')
                time_range = parts[0].strip()
                event_type = parts[2].strip()
                date_str   = time_range[:10]
                start_str, end_str = time_range[11:].split('-')
                start = pd.to_datetime(date_str + ' ' + start_str, format='%d.%m.%Y %H:%M:%S,%f')
                end   = pd.to_datetime(date_str + ' ' + end_str,   format='%d.%m.%Y %H:%M:%S,%f')
                if end < start:
                    end += pd.Timedelta(days=1)
                events.append({'start': start, 'end': end, 'event': event_type})
            except:
                pass
    return pd.DataFrame(events)

def plot_participant(folder, out_dir):
    flow_f   = find_file(folder, ['flow'],   exclude=['events'])
    thorac_f = find_file(folder, ['thorac'])
    spo2_f   = find_file(folder, ['spo2'])
    events_f = find_file(folder, ['flow', 'events'])

    if not all([flow_f, thorac_f, spo2_f, events_f]):
        print(f"missing files in {folder}, skipping")
        return

    flow   = parse_signal(flow_f)
    thorac = parse_signal(thorac_f)
    spo2   = parse_signal(spo2_f)
    events = parse_events(events_f)
    pid    = os.path.basename(folder)

    flow_ds   = flow.iloc[::32]
    thorac_ds = thorac.iloc[::32]
    spo2_ds   = spo2.iloc[::4]

    colors = {
        'Hypopnea':          '#FF8C00',
        'Obstructive Apnea': '#CC0000',
        'Central Apnea':     '#9900CC',
        'Mixed Apnea':       '#0066CC',
    }

    fig, axes = plt.subplots(3, 1, figsize=(20, 12), sharex=True)
    fig.suptitle(f'Sleep Study - Participant {pid}', fontsize=16, fontweight='bold', y=0.98)

    for sig, label, color, ax in [
        (flow_ds,   'Nasal Airflow',     'steelblue', axes[0]),
        (thorac_ds, 'Thoracic Movement', 'seagreen',  axes[1]),
        (spo2_ds,   'SpO2 (%)',          'crimson',   axes[2]),
    ]:
        t0  = sig.index[0]
        hrs = (sig.index - t0).total_seconds() / 3600.0
        ax.plot(hrs, sig.values, color=color, linewidth=0.4, alpha=0.85)
        ax.set_ylabel(label, fontsize=11)
        ax.grid(True, alpha=0.3)
        for _, ev in events.iterrows():
            s = (ev['start'] - t0).total_seconds() / 3600.0
            e = (ev['end']   - t0).total_seconds() / 3600.0
            ax.axvspan(s, e, alpha=0.35, color=colors.get(ev['event'], '#888'), linewidth=0)

    axes[2].set_xlabel('Time (hours from start)', fontsize=11)
    patches = [mpatches.Patch(color=c, label=ev, alpha=0.6) for ev, c in colors.items()]
    fig.legend(handles=patches, loc='lower center', ncol=4, fontsize=10,
               title='Breathing Events', title_fontsize=10, bbox_to_anchor=(0.5, 0.01))
    plt.tight_layout(rect=[0, 0.05, 1, 0.97])

    os.makedirs(out_dir, exist_ok=True)
    out = os.path.join(out_dir, f'{pid}_visualization.pdf')
    fig.savefig(out, format='pdf', dpi=150, bbox_inches='tight')
    print(f"saved → {out}")
    plt.close()

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('-name', required=True, help='path to participant folder e.g. Data/AP01')
    args    = parser.parse_args()
    folder  = args.name
    out_dir = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(folder))), 'Visualizations')
    plot_participant(folder, out_dir)
'''

# ── create_dataset.py ──────────────────────────────────────────
create_py = '''
import os, argparse
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt

def find_file(folder, keywords, exclude=None):
    for fname in os.listdir(folder):
        f = fname.lower()
        if all(k.lower() in f for k in keywords):
            if exclude and any(e.lower() in f for e in exclude):
                continue
            return os.path.join(folder, fname)
    return None

def parse_signal(filepath):
    timestamps, values = [], []
    reading = False
    with open(filepath, 'r', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if line.lower() == 'data:':
                reading = True
                continue
            if reading and ';' in line:
                parts = line.split(';')
                try:
                    ts  = pd.to_datetime(parts[0].strip(), format='%d.%m.%Y %H:%M:%S,%f')
                    val = float(parts[1].strip())
                    timestamps.append(ts)
                    values.append(val)
                except:
                    pass
    return pd.Series(values, index=pd.DatetimeIndex(timestamps), name='value')

def parse_events(filepath):
    events = []
    with open(filepath, 'r', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if ';' not in line or '-' not in line.split(';')[0]:
                continue
            try:
                parts      = line.split(';')
                time_range = parts[0].strip()
                event_type = parts[2].strip()
                date_str   = time_range[:10]
                start_str, end_str = time_range[11:].split('-')
                start = pd.to_datetime(date_str + ' ' + start_str, format='%d.%m.%Y %H:%M:%S,%f')
                end   = pd.to_datetime(date_str + ' ' + end_str,   format='%d.%m.%Y %H:%M:%S,%f')
                if end < start:
                    end += pd.Timedelta(days=1)
                events.append({'start': start, 'end': end, 'event': event_type})
            except:
                pass
    return pd.DataFrame(events)

def bandpass_filter(sig, fs, lo=0.17, hi=0.4, order=4):
    nyq  = fs / 2.0
    b, a = butter(order, [lo/nyq, hi/nyq], btype='band')
    return filtfilt(b, a, sig)

def get_label(win_start, win_end, events_df):
    duration = (win_end - win_start).total_seconds()
    for _, ev in events_df.iterrows():
        overlap = (min(win_end, ev['end']) - max(win_start, ev['start'])).total_seconds()
        if overlap > 0.5 * duration:
            return ev['event']
    return 'Normal'

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('-in_dir',  required=True)
    parser.add_argument('-out_dir', required=True)
    args = parser.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    all_rows = []

    for pid in sorted(os.listdir(args.in_dir)):
        folder = os.path.join(args.in_dir, pid)
        if not os.path.isdir(folder):
            continue
        print(f"processing {pid}...")

        flow_f   = find_file(folder, ['flow'],   exclude=['events'])
        thorac_f = find_file(folder, ['thorac'])
        spo2_f   = find_file(folder, ['spo2'])
        events_f = find_file(folder, ['flow', 'events'])

        if not all([flow_f, thorac_f, spo2_f, events_f]):
            print(f"  missing files, skipping")
            continue

        flow   = parse_signal(flow_f)
        thorac = parse_signal(thorac_f)
        spo2   = parse_signal(spo2_f)
        events = parse_events(events_f)

        flow_f2   = bandpass_filter(flow.values,   fs=32)
        thorac_f2 = bandpass_filter(thorac.values, fs=32)
        spo2_f2   = bandpass_filter(spo2.values,   fs=4)

        win, step, n = 960, 480, 0
        for i in range(0, len(flow_f2) - win + 1, step):
            ws = flow.index[i]
            we = flow.index[i + win - 1]
            si = i // 8
            se = si + 120
            if se > len(spo2_f2):
                break
            label = get_label(ws, we, events)
            row = {'participant': pid, 'win_start': str(ws), 'label': label}
            for j, v in enumerate(flow_f2[i:i+win]):   row[f'flow_{j}']   = v
            for j, v in enumerate(thorac_f2[i:i+win]): row[f'thorac_{j}'] = v
            for j, v in enumerate(spo2_f2[si:se]):     row[f'spo2_{j}']   = v
            all_rows.append(row)
            n += 1
        print(f"  {pid}: {n} windows")

    df = pd.DataFrame(all_rows)
    df.to_csv(os.path.join(args.out_dir, 'breathing_dataset.csv'), index=False)
    print(f"saved {len(df)} windows to {args.out_dir}/breathing_dataset.csv")
'''

# ── train_model.py ─────────────────────────────────────────────
train_py = '''
import os, argparse
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

class CNN1D(nn.Module):
    def __init__(self, num_classes):
        super(CNN1D, self).__init__()
        self.conv_block = nn.Sequential(
            nn.Conv1d(3, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(4),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(4),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 15, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.conv_block(x))

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('-dataset', default='Dataset/breathing_dataset.csv')
    args = parser.parse_args()

    df = pd.read_csv(args.dataset)
    df['label'] = df['label'].replace({'Body event': 'Other', 'Mixed Apnea': 'Other'})

    le = LabelEncoder()
    df['label_enc'] = le.fit_transform(df['label'])

    flow_cols   = [c for c in df.columns if c.startswith('flow_')]
    thorac_cols = [c for c in df.columns if c.startswith('thorac_')]
    spo2_cols   = [c for c in df.columns if c.startswith('spo2_')]

    flow_data   = df[flow_cols].values.astype(np.float32)
    thorac_data = df[thorac_cols].values.astype(np.float32)
    spo2_up     = np.repeat(df[spo2_cols].values.astype(np.float32), 8, axis=1)
    X           = np.stack([flow_data, thorac_data, spo2_up], axis=1)
    y           = df['label_enc'].values.astype(np.int64)
    participants = df['participant'].values

    device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    all_pids = sorted(df['participant'].unique())
    results  = []

    for test_pid in all_pids:
        print(f"\\nFold: test on {test_pid}")
        train_mask = participants != test_pid
        test_mask  = participants == test_pid

        X_train = torch.tensor(X[train_mask])
        y_train = torch.tensor(y[train_mask])
        X_test  = torch.tensor(X[test_mask])
        y_test  = torch.tensor(y[test_mask])

        classes = np.unique(y[train_mask])
        cw      = compute_class_weight('balanced', classes=classes, y=y[train_mask])
        weights = np.ones(len(le.classes_))
        weights[classes] = cw
        class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

        loader    = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
        model     = CNN1D(num_classes=len(le.classes_)).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

        for epoch in range(15):
            model.train()
            for xb, yb in loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                criterion(model(xb), yb).backward()
                optimizer.step()
            scheduler.step()

        model.eval()
        with torch.no_grad():
            preds = model(X_test.to(device)).argmax(dim=1).cpu().numpy()

        acc  = accuracy_score(y_test, preds)
        prec = precision_score(y_test, preds, average="macro", zero_division=0)
        rec  = recall_score(y_test, preds, average="macro", zero_division=0)
        cm   = confusion_matrix(y_test, preds)

        print(f"  Accuracy : {acc:.4f}")
        print(f"  Precision: {prec:.4f}")
        print(f"  Recall   : {rec:.4f}")
        print(f"  Confusion Matrix:\\n{cm}")
        results.append({'fold': test_pid, 'accuracy': acc, 'precision': prec, 'recall': rec})

    print("\\n=== Summary ===")
    for r in results:
        print(f"{r['fold']}  acc={r['accuracy']:.4f}  prec={r['precision']:.4f}  rec={r['recall']:.4f}")
    print(f"Mean Accuracy : {np.mean([r['accuracy']  for r in results]):.4f}")
    print(f"Mean Precision: {np.mean([r['precision'] for r in results]):.4f}")
    print(f"Mean Recall   : {np.mean([r['recall']    for r in results]):.4f}")
    print(f"Classes: {list(le.classes_)}")
'''

# ── Save all scripts ───────────────────────────────────────────
scripts_dir = '/content/internship/internship/scripts'
os.makedirs(scripts_dir, exist_ok=True)

with open(scripts_dir + '/vis.py',            'w') as f: f.write(vis_py.strip())
with open(scripts_dir + '/create_dataset.py', 'w') as f: f.write(create_py.strip())
with open(scripts_dir + '/train_model.py',    'w') as f: f.write(train_py.strip())

print("scripts saved:")
for s in os.listdir(scripts_dir):
    print(f"  scripts/{s}")

scripts saved:
  scripts/create_dataset.py
  scripts/train_model.py
  scripts/vis.py


In [ ]:
readme = '''# Sleep Apnea Detection from Physiological Signals

A machine learning pipeline for detecting breathing irregularities during sleep using nasal airflow, thoracic movement, and SpO₂ signals.

## Dataset
Overnight sleep recordings (8 hours) from 5 participants. Each participant folder contains:
- Nasal airflow signal (32 Hz)
- Thoracic movement signal (32 Hz)
- SpO₂ signal (4 Hz)
- Flow events file (annotated breathing irregularities)
- Sleep profile file (sleep stages)

## Project Structure
```
Project/
├── Data/               # raw participant signal files
├── Visualizations/     # generated PDF plots per participant
├── Dataset/            # processed labeled windows (CSV)
├── scripts/
│   ├── vis.py              # visualization script
│   ├── create_dataset.py   # preprocessing + dataset creation
│   └── train_model.py      # CNN training + LOPO evaluation
├── README.md
└── requirements.txt
```

## Usage

**Step 1 — Generate visualization for a participant:**
```bash
python scripts/vis.py -name Data/AP01
```

**Step 2 — Create the dataset:**
```bash
python scripts/create_dataset.py -in_dir Data -out_dir Dataset
```

**Step 3 — Train and evaluate:**
```bash
python scripts/train_model.py -dataset Dataset/breathing_dataset.csv
```

## Methods

### Preprocessing
- Bandpass filter (0.17–0.4 Hz) applied to all signals to retain breathing frequency range
- Signals split into 30-second windows with 50% overlap
- At 32 Hz: window = 960 samples, step = 480 samples
- SpO₂ upsampled from 120 to 960 samples to match other channels

### Labeling
- Each window labeled based on overlap with annotated events
- If >50% of window overlaps with a breathing event → that event label
- Otherwise → Normal

### Model
- 1D CNN with 3 input channels (flow, thoracic, SpO₂)
- Architecture: 3x Conv1D → BatchNorm → ReLU → MaxPool blocks, followed by fully connected layers
- Class weights used to handle imbalance

### Evaluation
- Leave-One-Participant-Out (LOPO) cross-validation across 5 participants
- Metrics: Accuracy, Precision, Recall, Confusion Matrix

## Results

| Fold | Accuracy | Precision | Recall |
|------|----------|-----------|--------|
| AP01 | 0.6762   | 0.3696    | 0.6229 |
| AP02 | 0.7954   | 0.3899    | 0.5241 |
| AP03 | 0.3626   | 0.2513    | 0.1690 |
| AP04 | 0.8333   | 0.2855    | 0.3033 |
| AP05 | 0.5857   | 0.2909    | 0.2946 |
| **Mean** | **0.6506 ± 0.17** | **0.3175 ± 0.05** | **0.3828 ± 0.17** |

Classes: Hypopnea, Normal, Obstructive Apnea, Other

**Note:** Low precision/recall is expected due to heavy class imbalance
(~91% Normal windows). Class weights were used to partially address this.

## Note on AI Tool Usage-- > i have used AI (claude) to write parts of the code .I am capable of explaining any part if required .
'''

requirements = '''numpy
pandas
scipy
matplotlib
torch
scikit-learn
'''

base = '/content/internship/internship'
with open(base + '/README.md',        'w') as f: f.write(readme.strip())
with open(base + '/requirements.txt', 'w') as f: f.write(requirements.strip())

print("README.md saved")
print("requirements.txt saved")

# verify final structure
print("\nFinal project structure:")
for root, dirs, files in os.walk(base):
    # skip Data folder contents to keep it clean
    dirs[:] = [d for d in sorted(dirs) if d != '__pycache__']
    level   = root.replace(base, '').count(os.sep)
    indent  = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if 'Data' in root and root != base + '/Data':
        continue
    for fname in sorted(files):
        print(f"{indent}  {fname}")

README.md saved
requirements.txt saved

Final project structure:
internship/
  README.md
  requirements.txt
  Data/
    AP01/
    AP02/
    AP03/
    AP04/
    AP05/
  Dataset/
  Visualizations/
    AP01_visualization.pdf
    AP02_visualization.pdf
    AP03_visualization.pdf
    AP04_visualization.pdf
    AP05_visualization.pdf
  scripts/
    create_dataset.py
    train_model.py
    vis.py
